# Prerequisites

In [ ]:
# get data for labs
!wget -nc -O around_the_world_in_80_days.txt https://www.gutenberg.org/ebooks/103.txt.utf-8

# 1. Word Count

Instructions:  
For each cell marked "double-click and add explanation here" please answer the question in your own words.  
In the section where you complete the code to perform basic nlp text cleaning and exploration tasks, the goal is to chain all of the transformations together in a single function. For learning and exploration purposes, it is acceptable to have each step seperate, but the last cell in this section should be one function with all transformations chained together.  
For steps c and f, it is acceptable to use your favorite chatbot to generate a list of common stop words (c) and punctuation (e) for use in the code. As these are common steps in nlp/text processing tasks, there are pleanty of libraries to help with this such as nltk, but there is no need to import extra dependencies for this lab unless you are already familiar with working with them.

In [ ]:
# start a spark session and create spark context for making rdd
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("word_count") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
# Defind the rdd
# Le fichier a ete telecharge par wget dans le repertoire de travail
# courant (monte sur $PROJECT_DIR dans le conteneur Docker), on utilise
# donc un chemin relatif et non '/content/...' (qui est specifique a
# Google Colab).
rdd = sc.textFile('around_the_world_in_80_days.txt')

# rdd_2 (version francaise) n'est utile que pour l'extension optionnelle
# "Where to go from here". Il faut d'abord la telecharger, par exemple :
# !wget -nc -O le_tour_du_monde_en_quatre_vingts_jours.txt \
#     https://www.gutenberg.org/ebooks/46541.txt.utf-8
# rdd_2 = sc.textFile('le_tour_du_monde_en_quatre_vingts_jours.txt')

In [ ]:
# view the first x lines of the rdd
rdd.take(20)

In [ ]:
# example lambda function
words = rdd.flatMap(lambda lines: lines.split(' '))

In [ ]:
# Note and explain the output of the below command
words

Le resultat affiche n'est pas la liste des mots, mais une representation
de l'objet RDD lui-meme, quelque chose comme
`PythonRDD[7] at RDD at PythonRDD.scala:53`. En effet, `flatMap` est une
**transformation**, et Spark applique une **evaluation paresseuse (lazy
evaluation)** : aucune transformation n'est reellement executee tant
qu'aucune **action** (comme `collect()`, `take()`, `count()`...) n'est
appelee. Afficher `words` montre donc seulement le plan de calcul prevu,
pas son resultat concret.

Cette evaluation paresseuse est un choix de conception important de
Spark : tant qu'aucune action n'a ete appelee, Spark peut construire le
graphe complet des transformations demandees (le DAG) et l'optimiser
globalement (fusionner des etapes, eviter des calculs inutiles,
minimiser les echanges de donnees entre executeurs) avant de l'executer
reellement sur le cluster. C'est une difference fondamentale avec une
liste ou un generateur Python classique, ou chaque instruction
s'execute immediatement des qu'elle est ecrite.

In [ ]:
# Note and explain the output of the following command, focusing on the difference with the
# above command
words.collect()

Contrairement a l'affichage precedent, `collect()` est une **action** :
elle declenche reellement l'execution de toutes les transformations en
attente (ici le `flatMap`) sur les executeurs, puis rapatrie
l'integralite du resultat sous forme de liste Python standard sur le
driver. C'est pour cela qu'on voit maintenant la vraie liste des mots
du texte, et non plus une reference abstraite au RDD. A noter :
`collect()` charge tout le resultat en memoire sur le driver, ce qui
peut etre risque sur de gros volumes de donnees ; pour explorer un RDD
sans danger, on prefere generalement `take(n)`.

In [ ]:
# nicer print
for w in words.collect():
    print(w)

In [ ]:
# Print first x words
words.take(20)

In [ ]:
# Use cell magic command to help understand what the rdd.flatMap function is doing in the next cell.
# Insert a text/markdown cell and explain in your own words.
rdd.flatMap?

D'apres la docstring affichee par `rdd.flatMap?`, `flatMap(f)` applique
la fonction `f` a chaque element du RDD (comme le ferait `map`), puis
**aplatit** ("flatten") les resultats en une seule sequence plate,
plutot que de renvoyer une liste de listes. Ici, `f` decoupe chaque
ligne du livre en mots (`lines.split(' ')`) : avec `map`, on obtiendrait
un RDD ou chaque element est une liste de mots (un element par ligne).
Avec `flatMap`, toutes ces sous-listes sont fusionnees en un seul RDD
"plat" ou chaque element est directement un mot individuel.

In [ ]:
# Initialize a word counter by creating a tuple with word and cound of 1
words = rdd.flatMap(lambda lines: lines.split(' ')) \
                    .map(lambda word: (word, 1))

for w in words.collect():
    print(w)

In [ ]:
# a. count the occurence of each word
word_counts = (
    rdd.flatMap(lambda line: line.split(' '))
       .map(lambda word: (word, 1))
       .reduceByKey(lambda a, b: a + b)
)
word_counts.take(20)

In [ ]:
# b. a common first step in text analysis, change all capital letters to lower case
word_counts_lc = (
    rdd.flatMap(lambda line: line.split(' '))
       .map(lambda word: (word.lower(), 1))
       .reduceByKey(lambda a, b: a + b)
)
word_counts_lc.take(20)

In [ ]:
# c. eliminate the stop words.
# Liste de stop words anglais courants (generee avec l'aide d'un
# chatbot, ce qui est autorise pour cette etape par la consigne du lab).
STOP_WORDS = {
    'a', 'about', 'above', 'after', 'again', 'against', 'all', 'am', 'an',
    'and', 'any', 'are', 'as', 'at', 'be', 'because', 'been', 'before',
    'being', 'below', 'between', 'both', 'but', 'by', 'can', 'did', 'do',
    'does', 'doing', 'down', 'during', 'each', 'few', 'for', 'from',
    'further', 'had', 'has', 'have', 'having', 'he', 'her', 'here',
    'hers', 'herself', 'him', 'himself', 'his', 'how', 'i', 'if', 'in',
    'into', 'is', 'it', 'its', 'itself', 'just', 'me', 'more', 'most',
    'my', 'myself', 'no', 'nor', 'not', 'now', 'of', 'off', 'on', 'once',
    'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over',
    'own', 'same', 'she', 'should', 'so', 'some', 'such', 'than', 'that',
    'the', 'their', 'theirs', 'them', 'themselves', 'then', 'there',
    'these', 'they', 'this', 'those', 'through', 'to', 'too', 'under',
    'until', 'up', 'very', 'was', 'we', 'were', 'what', 'when', 'where',
    'which', 'while', 'who', 'whom', 'why', 'will', 'with', 'you',
    'your', 'yours', 'yourself', 'yourselves',
}

word_counts_no_stop = (
    rdd.flatMap(lambda line: line.split(' '))
       .map(lambda word: word.lower())
       .filter(lambda word: word not in STOP_WORDS)
       .map(lambda word: (word, 1))
       .reduceByKey(lambda a, b: a + b)
)
word_counts_no_stop.take(20)

In [ ]:
# d. sort in alphabetical order
word_counts_alpha = word_counts_no_stop.sortByKey()
word_counts_alpha.take(20)

In [ ]:
# e. sort descending by word frequency
word_counts_freq_desc = word_counts_no_stop.sortBy(
    lambda pair: pair[1], ascending=False
)
word_counts_freq_desc.take(20)

In [ ]:
# f. remove punctuations and blank spaces
import string

# Jeu de caracteres de ponctuation standard (genere avec l'aide d'un
# chatbot, autorise pour cette etape par la consigne du lab).
PUNCTUATION = string.punctuation  # !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~

word_counts_clean = (
    rdd.flatMap(lambda line: line.split(' '))
       .map(lambda word: word.lower().strip(PUNCTUATION))
       .filter(lambda word: word != '' and word not in STOP_WORDS)
       .map(lambda word: (word, 1))
       .reduceByKey(lambda a, b: a + b)
)
word_counts_clean.take(20)

In [ ]:
# Fonction unique qui enchaine toutes les transformations ci-dessus :
# comptage (a), minuscules (b), retrait des stop words (c), tri (d, e)
# et retrait de la ponctuation/espaces vides (f).
def word_count_pipeline(text_rdd, stop_words, punctuation):
    return (
        text_rdd.flatMap(lambda line: line.split(' '))
                .map(lambda word: word.lower().strip(punctuation))
                .filter(lambda word: word != '' and word not in stop_words)
                .map(lambda word: (word, 1))
                .reduceByKey(lambda a, b: a + b)
                .sortBy(lambda pair: pair[1], ascending=False)
    )


final_word_counts = word_count_pipeline(rdd, STOP_WORDS, string.punctuation)
final_word_counts.take(20)

# 2. What does the following cell block do?
Comment the code below line by line after the provided hash-tag. You should be able to explain each line while respecting the pep8 style guide of 79 characters or less per line!

In [ ]:
 # Create an RDD of tuples (name, age)
dataRDD = sc.parallelize([("Brooke", 20), ("Denny", 31), ("Jules", 30),
("TD", 35), ("Brooke", 25)])

# Try to undestand what this code does (line by line)
agesRDD = (dataRDD
  # Pour chaque tuple (name, age), on cree (name, (age, 1)) : le 1
  # sert de compteur, utile pour calculer une moyenne par la suite.
  .map(lambda x: (x[0], (x[1], 1)))
  # On regroupe par cle (name) et on additionne separement les ages
  # et les compteurs entre eux :
  # (name, (somme_des_ages, nombre_d_occurrences)).
  .reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
  # On divise la somme des ages par le nombre d'occurrences pour
  # obtenir l'age moyen de chaque personne : (name, age_moyen).
  .map(lambda x: (x[0], x[1][0]/x[1][1])))

# Where to go from here

Further exploration for students who complete the lab before the end of the session or want to go further.

- perform eda on the original french version of the [book](https://www.gutenberg.org/ebooks/46541.txt.utf-8) and compare the two
- recomplete the exercises using a the docker install
- install java and spark directly onto host machine and either rexplore this notebook or perform eda on other data sets
- write a simple python timer function for seeing how quickly your rdd runs as written. change the order of the steps in order to make the rdd run as optimally as possible